Here we pick the best hyperparameters from the TriCondNet nested CV results, ready for training the final deployment ensemble.

### What's going on here

TriCondNet has three parts:

- a **classifier** that predicts whether a material is a metal or a semiconductor, and
- two **regressors** for conductivity, one trained on metals and one on semiconductors.

Each part was tuned separately during nested cross-validation, giving one `.pkl` result file per fold. This notebook goes through those files, finds the best-performing fold for each part, and saves its hyperparameters as small `*_hp.pkl` files. These are then used to train the final ensemble for inference.

In [1]:
import pandas as pandas 
from pathlib import Path
import sys 
import numpy as np
import pickle as pkl
import os

### Picking the best fold for each part

The three functions below work the same way, with different scoring:

- `classifier_block` reads every `classifier_fold_*` file and keeps the one with the highest balanced accuracy, then renames the LightGBM parameters (`bagging_fraction`, `feature_fraction`) to their scikit-learn equivalents (`subsample`, `colsample_bytree`).
- `semiconductor_block` and `metal_block` do the same but keep the fold with the lowest test MAE, saving its hyperparameters as-is.

`root` points at the nested-CV results folder. The three calls at the bottom run the selection and write out `classifier_hp.pkl`, `semi_hp.pkl`, and `metal_hp.pkl`.

In [ ]:
def classifier_block(hp_root, prefix="classifier_fold", export="classifier_hp.pkl"):
    metric, best_file = -np.inf, None
    for file_name in os.listdir(hp_root):
        if prefix in file_name:
            with open(hp_root / file_name, "rb") as f:
                file = pkl.load(f)
            balanced_accuracy = file["metrics"]["balanced_accuracy"]
            if balanced_accuracy > metric:
                metric = balanced_accuracy
                best_file = file

    bp = best_file["best_hp"]["best_params"]
    flat = {
        "n_estimators":      bp["n_estimators"],
        "learning_rate":     bp["learning_rate"],
        "max_depth":         bp["max_depth"],
        "num_leaves":        bp["num_leaves"],
        "min_child_samples": bp["min_child_samples"],
        "subsample":         bp["bagging_fraction"],  
        "colsample_bytree":  bp["feature_fraction"],  
        "reg_alpha":         bp["reg_alpha"],
        "reg_lambda":        bp["reg_lambda"],
        "class_weight":      "balanced",               # not in nested CV search; sensible default
        "min_split_gain":    0.0,                      # not in nested CV search; LGBM default
        "n_feat":            best_file["best_hp"]["best_n_feat"],
    }
    with open(export, "wb") as f:
        pkl.dump(flat, f)

def semiconductor_block(hp_root, prefix="SemiRegressor", export="semi_hp.pkl"):
    metric = np.inf
    for file_name in os.listdir(hp_root):
        if prefix in file_name: 
            file_path = hp_root / file_name
            with open(file_path, "rb") as f:
                file = pkl.load(f)
            test_mae = file["metrics"]["test_mae"]
            if test_mae<metric:
                metric = test_mae
                best_file = file
    with open(export, "wb") as f:
        pkl.dump(best_file["best_hp"], f)

def metal_block(hp_root, prefix="MetalRegressor", export="metal_hp.pkl"):
    metric = np.inf
    for file_name in os.listdir(hp_root):
        if prefix in file_name: 
            file_path = hp_root / file_name
            with open(file_path, "rb") as f:
                file = pkl.load(f)
            test_mae = file["metrics"]["test_mae"]
            if test_mae<metric:
                metric = test_mae
                best_file = file
    with open(export, "wb") as f:
        pkl.dump(best_file["best_hp"], f)

In [ ]:
Chosen_CBFV = "magpie" # Can change this depending on the Nested-CV results you are analyzing 
root = Path.cwd().parent / "nestedcv_demonstration" / "nested_cv_results"/ "TriCondNet"
Export_Path = Path.cwd() / "saved_hyperparameters" / f"{Chosen_CBFV}_hyperparameters"

classifier_block(root, export=Export_Path)
semiconductor_block(root, export=Export_Path)
metal_block(root, export=Export_Path)